In [2]:
import pandas as pd
from pathlib import Path

# Remonter les dossiers jusqu'à trouver data/raw (fonctionne où que soit le notebook)
racine = Path.cwd()
while not (racine / "data" / "raw").exists() and racine != racine.parent:
    racine = racine.parent

dossier_data = racine / "data" / "raw"
print(f"Dossier courant : {Path.cwd()}")
print(f"Dossier des données : {dossier_data}")

fichiers = sorted(dossier_data.glob("*.csv.gz"))
print(f"{len(fichiers)} fichiers trouvés")

df = pd.concat((pd.read_csv(f, low_memory=False) for f in fichiers), ignore_index=True)
print(df.shape)

Dossier courant : c:\Users\salwa\estimateur-immo-dvf\src\notebooks
Dossier des données : c:\Users\salwa\estimateur-immo-dvf\data\raw
30 fichiers trouvés
(1720509, 40)


In [3]:
df.columns.tolist()

['id_mutation',
 'date_mutation',
 'numero_disposition',
 'nature_mutation',
 'valeur_fonciere',
 'adresse_numero',
 'adresse_suffixe',
 'adresse_nom_voie',
 'adresse_code_voie',
 'code_postal',
 'code_commune',
 'nom_commune',
 'code_departement',
 'ancien_code_commune',
 'ancien_nom_commune',
 'id_parcelle',
 'ancien_id_parcelle',
 'numero_volume',
 'lot1_numero',
 'lot1_surface_carrez',
 'lot2_numero',
 'lot2_surface_carrez',
 'lot3_numero',
 'lot3_surface_carrez',
 'lot4_numero',
 'lot4_surface_carrez',
 'lot5_numero',
 'lot5_surface_carrez',
 'nombre_lots',
 'code_type_local',
 'type_local',
 'surface_reelle_bati',
 'nombre_pieces_principales',
 'code_nature_culture',
 'nature_culture',
 'code_nature_culture_speciale',
 'nature_culture_speciale',
 'surface_terrain',
 'longitude',
 'latitude']

valeur_fonciere est la valeur à prédire
Famille	Colonnes	Rôle pour nous
La vente	id_mutation, date_mutation, nature_mutation, valeur_fonciere	Le prix (valeur_fonciere) est la variable à prédire
L’adresse	adresse_*, code_postal, code_commune, nom_commune, code_departement	La localisation
Le bien	type_local, surface_reelle_bati, nombre_pieces_principales	Les caractéristiques principales
Le terrain	surface_terrain, nature_culture (sols, jardin, terres…)	Utile pour les maisons
Les lots	lot1_numero … lot5_surface_carrez, nombre_lots	Lots de copropriété (appartements)
GPS	longitude, latitude	Indispensable pour le modèle
Le reste	ancien_*, numero_volume, code_nature_culture_speciale…	On ignorera

Piège n°1 : une vente peut occuper plusieurs lignes. Une ligne correspond à un élément de la vente, pas à une vente. Si quelqu’un achète un appartement avec une cave et un parking pour 300 000 €, on a 3 lignes (Appartement, Dépendance, Dépendance), et chacune affiche 300 000 €. Les garder telles quelles revient à compter le prix trois fois.

Piège n°2 : un même bien peut être répété. Une maison dont le terrain a plusieurs natures de culture (une partie « sols », une partie « jardin ») apparaît sur une ligne par nature de culture. Si on ne fait pas attention, on croit qu’il y a deux maisons.

Stratégie : ne garder que les ventes qui portent sur exactement un logement (une maison ou un appartement), éventuellement avec des dépendances (caves, garages), et obtenir une ligne par vente. On écarte les ventes groupées, comme un immeuble entier ou un lot de 10 appartements, parce qu’on ne peut pas savoir combien vaut chaque logement.